In [1]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.entities import Environment, ManagedOnlineEndpoint, ManagedOnlineDeployment, CodeConfiguration, OnlineRequestSettings

# Azure ML workspace identifiers
subscription_id = 'ad017e07-0d6f-4cad-8669-16fd3fe4f33c'
resource_group = 'rg-dante.garciaperez-6665'
workspace = 'lucido'

# Get a handle to the workspace
ml_client = MLClient(DefaultAzureCredential(), subscription_id, resource_group, workspace)
model = ml_client.models.get('Qwen3-Embedding-06B', version='1')

In [2]:
# Image found at https://learn.microsoft.com/en-us/azure/machine-learning/concept-prebuilt-docker-images-inference?view=azureml-api-2
env = Environment(conda_file="conda.yaml", image="mcr.microsoft.com/azureml/minimal-py312-inference:latest")
ml_client.environments.create_or_update(env)

Environment({'arm_type': 'environment_version', 'latest_version': None, 'image': 'mcr.microsoft.com/azureml/minimal-py312-inference:latest', 'intellectual_property': None, 'is_anonymous': True, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'CliV2AnonymousEnvironment', 'description': None, 'tags': {}, 'properties': {'azureml.labels': ''}, 'print_as_yaml': False, 'id': '/subscriptions/ad017e07-0d6f-4cad-8669-16fd3fe4f33c/resourceGroups/rg-dante.garciaperez-6665/providers/Microsoft.MachineLearningServices/workspaces/lucido/environments/CliV2AnonymousEnvironment/versions/7a4511333a4c93e2f56d580927ae7202fa7e34527502c3155737b2678afef9ef', 'Resource__source_path': '', 'base_path': 'd:\\Projects\\summer-job\\azure_ml', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x000002238381C6B0>, 'serialize': <msrest.serialization.Serializer object at 0x000002238381C3B0>, 'version': '9ffdcef0bce71ca7025cdae9dd55911d247847124f6a2a506a28692babff606a', 'c

In [3]:
endpoint_name = 'lucido-embeddings-endpoint'

endpoint = ManagedOnlineEndpoint(
    name = endpoint_name, 
    description="Lucido embeddings endpoint",
    auth_mode="key"
)

In [4]:
deployment = ManagedOnlineDeployment(
    name="lucido-embeddings-deployment-1", 
    endpoint_name=endpoint_name,
    model=model,
    environment=env,
    code_configuration=CodeConfiguration(code="./", scoring_script="score.py"),
    instance_type="Standard_DS1_v2",
    instance_count=1,
    request_settings=OnlineRequestSettings(
        max_concurrent_requests_per_instance=10,
        request_timeout_ms=180_000,
        max_queue_wait_ms=300_000
    )
)

In [5]:
ml_client.online_endpoints.begin_create_or_update(endpoint)

In [ ]:
ml_client.online_deployments.begin_create_or_update(deployment)

Instance type Standard_DS1_v2 may be too small for compute resources. Minimum recommended compute SKU is Standard_DS3_v2 for general purpose endpoints. Learn more about SKUs here: https://learn.microsoft.com/azure/machine-learning/referencemanaged-online-endpoints-vm-sku-list
Check: endpoint lucido-embeddings-endpoint exists
Your file exceeds 100 MB. If you experience low speeds, latency, or broken connections, we recommend using the AzCopyv10 tool for this file transfer.

Example: azcopy copy 'D:\Projects\summer-job\azure_ml' 'https://lucido6964160932.blob.core.windows.net/8d3ff23c-62bf-4ba6-84e7-3bdd62180e0e-yygteppcfrfcsd8ocpcszubukn/azure_ml' 

See https://learn.microsoft.com/azure/storage/common/storage-use-azcopy-v10 for more information.
Uploading azure_ml (2399.1 MBs): 100%|##########| 2399095071/2399095071 [05:43<00:00, 6985464.40it/s]




................................................

In [14]:
# test the embeddings deployment with some sample data
response = ml_client.online_endpoints.invoke(
    endpoint_name=endpoint_name,
    deployment_name="lucido-embeddings-deployment-1",
    request_file="sample-request.json",
)

In [17]:
import json

json.loads(json.loads(response))['embeddings']

[0.01751134917140007,
 -0.025757869705557823,
 -0.009985002689063549,
 -0.017855986952781677,
 -0.02625991217792034,
 0.0076321689411997795,
 -0.043425366282463074,
 0.05734913423657417,
 -0.061020031571388245,
 -0.02741478942334652,
 -0.020892756059765816,
 0.018419578671455383,
 0.07078578323125839,
 -0.00836237147450447,
 -0.043117545545101166,
 0.0889332965016365,
 -0.004447998013347387,
 0.04913061857223511,
 0.10106978565454483,
 -0.06601919233798981,
 -0.0014161263825371861,
 0.007378630340099335,
 -0.030394064262509346,
 0.1045173779129982,
 -0.016085419803857803,
 -0.03364897891879082,
 -0.03965211287140846,
 0.07764439284801483,
 -0.0016175985801964998,
 -0.005080535542219877,
 -0.009543308056890965,
 0.07037637382745743,
 -0.015399486757814884,
 -0.0022649948950856924,
 -0.039451371878385544,
 -0.011068752035498619,
 0.006932846736162901,
 0.007218001410365105,
 -0.001285672769881785,
 0.05685941129922867,
 -0.02200254797935486,
 0.005372781306505203,
 0.05072609335184097,
 

In [1]:
import urllib.request
import json

data = {'text': 'The 9000 series are foolproof and perfectly incapable of error.'}
body = str.encode(json.dumps(data))
url = 'https://lucido-embeddings-endpoint.centralus.inference.ml.azure.com/score'

api_key = '9H5no7jfnlxmmBVLABPNaCeNrXnTewEtUwWDYOwRFYRBgHQXxvcWJQQJ99BGAAAAAAAAAAAAINFRAZML1JAF'
headers = {'Content-Type':'application/json', 'Accept': 'application/json', 'Authorization':('Bearer '+ api_key)}

req = urllib.request.Request(url, body, headers)
response = urllib.request.urlopen(req)
result = response.read()

In [6]:
json.loads(json.loads(result))['embeddings']

[0.014527362771332264,
 -0.03662661463022232,
 -0.015261939726769924,
 -0.047636840492486954,
 0.03608715161681175,
 0.059046465903520584,
 0.004910895135253668,
 -0.0007215093937702477,
 -0.04608237370848656,
 0.004987478721886873,
 -0.03173040598630905,
 0.016258129850029945,
 0.008901559747755527,
 -0.014093650504946709,
 -0.047746721655130386,
 0.12024498730897903,
 -0.025142382830381393,
 -0.005694492720067501,
 -0.013357080519199371,
 0.03903359919786453,
 -0.03019053302705288,
 0.035415589809417725,
 -0.02397599257528782,
 0.10827229171991348,
 -0.010557089000940323,
 0.013142680749297142,
 -0.038390472531318665,
 0.11850635707378387,
 0.03299202024936676,
 -0.04071016609668732,
 -0.03392605483531952,
 0.00297866971231997,
 -0.011745341122150421,
 -0.06650275737047195,
 -0.029238538816571236,
 -0.0207511093467474,
 0.04585200920701027,
 0.03739757835865021,
 -0.013164755888283253,
 -0.00022852959227748215,
 -0.013187027536332607,
 0.0028484456706792116,
 -0.02673959918320179,
 0